In [4]:
import pandas as pd

print("1. 데이터 불러오는 중... (파일 용량이 커서 1~2분 정도 걸릴 수 있습니다)")
# 주의: 다운로드 받으신 파일 이름이 정확히 일치하는지 확인해주세요!
file_path = "TCGA-LUAD.star_fpkm (1).tsv" 
df = pd.read_csv(file_path, sep='\t', index_col=0)

print("2. TCGA 바코드 규칙에 따라 암(Tumor)과 정상(Normal) 그룹 분류 중...")
tumor_cols = []
normal_cols = []

for col in df.columns:
    # TCGA 바코드는 'TCGA-44-6775-01A' 형태입니다.
    # 4번째 덩어리('01A')의 숫자가 01~09면 암, 10~19면 정상 조직입니다.
    parts = col.split('-')
    if len(parts) >= 4:
        sample_type = parts[3][:2]
        if sample_type.isdigit():
            if int(sample_type) < 10:
                tumor_cols.append(col)
            elif int(sample_type) >= 10:
                normal_cols.append(col)

print(f" -> 분류 완료: 암 조직 {len(tumor_cols)}개, 정상 조직 {len(normal_cols)}개")

print("3. 그룹별 평균 발현량 및 폴드 체인지(LogFC) 계산 중...")
# 다운받은 데이터는 이미 Log2 처리가 되어 있으므로, 공평하게 평균을 구한 뒤 빼줍니다.
tumor_mean = df[tumor_cols].mean(axis=1)
normal_mean = df[normal_cols].mean(axis=1)

# LogFC = 암 평균 - 정상 평균
df['Tumor_Mean'] = tumor_mean
df['Normal_Mean'] = normal_mean
df['LogFC'] = tumor_mean - normal_mean

print("4. 암에서 가장 많이 폭증한 상위 20개 핵심 유전자 추출 중...")
# LogFC가 높은 순(격차가 큰 순)으로 내림차순 정렬
top_genes = df.sort_values(by='LogFC', ascending=False).head(20)


# 결과만 깔끔하게 추려내기
result_df = top_genes[['Tumor_Mean', 'Normal_Mean', 'LogFC']]

print("\n==============================================================")
print("🏆 [TCGA 암 조직 vs 암 주변 조직] 발현량 급증 Top 20 유전자")
print("==============================================================")
print(result_df)

# 나중에 API로 이름을 찾거나 AI 학습을 위해 엑셀(CSV)로 저장해둡니다.
result_df.to_csv("My_Top20_Cancer_Markers.csv")
print("\n✅ 분석 완료! 'My_Top20_Cancer_Markers.csv' 파일이 저장되었습니다.")

1. 데이터 불러오는 중... (파일 용량이 커서 1~2분 정도 걸릴 수 있습니다)
2. TCGA 바코드 규칙에 따라 암(Tumor)과 정상(Normal) 그룹 분류 중...
 -> 분류 완료: 암 조직 530개, 정상 조직 59개
3. 그룹별 평균 발현량 및 폴드 체인지(LogFC) 계산 중...
4. 암에서 가장 많이 폭증한 상위 20개 핵심 유전자 추출 중...

🏆 [TCGA 암 조직 vs 암 주변 조직] 발현량 급증 Top 20 유전자
                    Tumor_Mean  Normal_Mean     LogFC
Ensembl_ID                                           
ENSG00000118785.14    6.835151     2.522476  4.312675
ENSG00000143320.9     6.164493     1.981082  4.183412
ENSG00000147689.16    3.752873     0.264830  3.488042
ENSG00000164266.11    4.377413     0.925584  3.451829
ENSG00000105388.16    5.654209     2.351758  3.302451
ENSG00000164932.13    4.999687     1.715547  3.284141
ENSG00000099953.10    3.727714     0.555199  3.172515
ENSG00000211892.4     5.978905     2.840092  3.138814
ENSG00000183010.17    4.762264     1.648184  3.114081
ENSG00000175063.17    4.412802     1.317161  3.095641
ENSG00000163993.7     6.629272     3.618191  3.011081
ENSG00000179913.11    3.824156     0.826452  2.

/var/folders/6g/7fmgk_ln1979hm_m4d8kd8g40000gn/T/ipykernel_1666/1141065087.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Tumor_Mean'] = tumor_mean
/var/folders/6g/7fmgk_ln1979hm_m4d8kd8g40000gn/T/ipykernel_1666/1141065087.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Normal_Mean'] = normal_mean
/var/folders/6g/7fmgk_ln1979hm_m4d8kd8g40000gn/T/ipykernel_1666/1141065087.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor p

In [3]:
import pandas as pd
import requests

print("1. 저장해둔 상위 20개 마커 데이터를 불러옵니다...")
# 아까 저장한 파일 불러오기
df = pd.read_csv("My_Top20_Cancer_Markers.csv")
# 첫 번째 열(Ensembl ID)의 이름을 명확하게 지정
df.rename(columns={df.columns[0]: 'Ensembl_ID'}, inplace=True)

print("2. Ensembl ID 정제 중... (버전 꼬리표 떼기)")
# TCGA 데이터의 ID는 'ENSG00000000003.15' 처럼 뒤에 소수점(버전)이 붙어있습니다.
# 검색을 위해 소수점 뒤의 숫자를 잘라냅니다.
df['Clean_ID'] = df['Ensembl_ID'].apply(lambda x: x.split('.')[0])
clean_id_list = df['Clean_ID'].tolist()

print("3. 국제 유전자 데이터베이스(MyGene.info)에 진짜 이름 물어보는 중...")
url = "https://mygene.info/v3/query"
params = {
    'q': ','.join(clean_id_list),
    'scopes': 'ensembl.gene',
    'fields': 'symbol,name', # 유전자 심볼(짧은 이름)과 풀네임(설명)을 요청
    'species': 'human'
}

response = requests.post(url, data=params)
gene_data = response.json()

print("4. 결과표 합치는 중...")
# 검색 결과를 딕셔너리로 정리
gene_dict = {}
for item in gene_data:
    if 'query' in item and 'symbol' in item:
        gene_dict[item['query']] = {
            'Symbol': item.get('symbol', 'Unknown'),
            'Description': item.get('name', 'Unknown')
        }

# 원래 데이터프레임에 새로운 열 추가
df['Gene_Symbol'] = df['Clean_ID'].map(lambda x: gene_dict.get(x, {}).get('Symbol', 'Unknown'))
df['Gene_Description'] = df['Clean_ID'].map(lambda x: gene_dict.get(x, {}).get('Description', 'Unknown'))

# 보기 좋게 열 순서 정리
final_df = df[['Ensembl_ID', 'Gene_Symbol', 'Gene_Description', 'Tumor_Mean', 'Normal_Mean', 'LogFC']]

print("\n==============================================================")
print("✨ [해독 완료] 암 세포에서 폭증한 Top 유전자 정체")
print("==============================================================")
# 보기 좋게 출력
print(final_df[['Gene_Symbol', 'Gene_Description', 'LogFC']].head(10))

# 최종 완성본 엑셀(CSV) 저장
final_df.to_csv("Final_Translated_Cancer_Markers.csv", index=False)
print("\n✅ 해독 완료! 'Final_Translated_Cancer_Markers.csv' 파일로 저장되었습니다.")

1. 저장해둔 상위 20개 마커 데이터를 불러옵니다...
2. Ensembl ID 정제 중... (버전 꼬리표 떼기)
3. 국제 유전자 데이터베이스(MyGene.info)에 진짜 이름 물어보는 중...
4. 결과표 합치는 중...

✨ [해독 완료] 암 세포에서 폭증한 Top 유전자 정체
  Gene_Symbol                                   Gene_Description     LogFC
0        SPP1                          secreted phosphoprotein 1  4.312675
1      CRABP2           cellular retinoic acid binding protein 2  4.183412
2      SACK1A                scaffolding CK1 anchoring protein A  3.488042
3      SPINK1            serine peptidase inhibitor Kazal type 1  3.451829
4     CEACAM5                       CEA cell adhesion molecule 5  3.302451
5      CTHRC1          collagen triple helix repeat containing 1  3.284141
6       MMP11                         matrix metallopeptidase 11  3.172515
7       IGHG4  immunoglobulin heavy constant gamma 4 (G4m mar...  3.138814
8       PYCR1                pyrroline-5-carboxylate reductase 1  3.114081
9       UBE2C                  ubiquitin conjugating enzyme E2 C  3.095641

✅ 해독 완료! 'Fi